# Customer Segmentation Clustering

Use this notebook when the case study asks for unsupervised customer segmentation. It is designed for structured tabular data and supports 2D and 3D cluster visualizations. If there are many features, the notebook projects them with PCA for plotting.


## Workflow

1. Load customer-level data
2. Select segmentation features
3. Scale and encode inputs
4. Compare clustering candidates
5. Visualize clusters in 2D and 3D
6. Profile clusters and write business actions


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)
plt.style.use('ggplot')


In [ ]:
DATA_PATH = Path('data.csv')
ID_COLUMN = 'customer_id'

# Optionally define these manually if you do not want the notebook to auto-detect.
FEATURE_COLUMNS = []

df = pd.read_csv(DATA_PATH)
display(df.head())
print(df.shape)


In [ ]:
if FEATURE_COLUMNS:
    segmentation_df = df[FEATURE_COLUMNS].copy()
else:
    excluded = {ID_COLUMN}
    segmentation_df = df[[c for c in df.columns if c not in excluded]].copy()

numeric_features = segmentation_df.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in segmentation_df.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ],
    remainder='drop',
)

X = preprocessor.fit_transform(segmentation_df)
if hasattr(X, 'toarray'):
    X_dense = X.toarray()
else:
    X_dense = X

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Model matrix shape:', X_dense.shape)


In [ ]:
results = []

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_dense)
    score = silhouette_score(X_dense, labels)
    results.append({'model': 'KMeans', 'setting': f'k={k}', 'silhouette': score})

for k in range(2, 7):
    model = AgglomerativeClustering(n_clusters=k)
    labels = model.fit_predict(X_dense)
    score = silhouette_score(X_dense, labels)
    results.append({'model': 'Agglomerative', 'setting': f'k={k}', 'silhouette': score})

dbscan = DBSCAN(eps=1.5, min_samples=3)
dbscan_labels = dbscan.fit_predict(X_dense)
valid_mask = dbscan_labels != -1
if valid_mask.sum() > 1 and len(set(dbscan_labels[valid_mask])) > 1:
    dbscan_score = silhouette_score(X_dense[valid_mask], dbscan_labels[valid_mask])
    results.append({'model': 'DBSCAN', 'setting': 'eps=1.5,min_samples=3', 'silhouette': dbscan_score})

comparison = pd.DataFrame(results).sort_values('silhouette', ascending=False)
display(comparison)


In [ ]:
SELECTED_MODEL = 'KMeans'
SELECTED_K = 3

if SELECTED_MODEL == 'KMeans':
    final_model = KMeans(n_clusters=SELECTED_K, random_state=42, n_init=10)
    cluster_labels = final_model.fit_predict(X_dense)
elif SELECTED_MODEL == 'Agglomerative':
    final_model = AgglomerativeClustering(n_clusters=SELECTED_K)
    cluster_labels = final_model.fit_predict(X_dense)
else:
    final_model = DBSCAN(eps=1.5, min_samples=3)
    cluster_labels = final_model.fit_predict(X_dense)

clustered_df = df.copy()
clustered_df['cluster'] = cluster_labels
display(clustered_df.head())
display(clustered_df['cluster'].value_counts(dropna=False).sort_index())


In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
coords_2d = pca_2d.fit_transform(X_dense)

plot_df_2d = pd.DataFrame(coords_2d, columns=['pc1', 'pc2'])
plot_df_2d['cluster'] = cluster_labels

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(plot_df_2d['pc1'], plot_df_2d['pc2'], c=plot_df_2d['cluster'], cmap='tab10', alpha=0.8)
ax.set_title('Customer Segments in 2D PCA Space')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
legend = ax.legend(*scatter.legend_elements(), title='Cluster')
ax.add_artist(legend)
plt.tight_layout()
plt.show()


In [ ]:
pca_3d = PCA(n_components=3, random_state=42)
coords_3d = pca_3d.fit_transform(X_dense)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(coords_3d[:, 0], coords_3d[:, 1], coords_3d[:, 2], c=cluster_labels, cmap='tab10', alpha=0.8)
ax.set_title('Customer Segments in 3D PCA Space')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
fig.colorbar(scatter, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()


In [ ]:
profile_cols = [c for c in clustered_df.columns if c != ID_COLUMN and c != 'cluster']
numeric_profile_cols = clustered_df[profile_cols].select_dtypes(include=np.number).columns.tolist()
categorical_profile_cols = [c for c in profile_cols if c not in numeric_profile_cols]

if numeric_profile_cols:
    cluster_profile_numeric = clustered_df.groupby('cluster')[numeric_profile_cols].mean().round(2)
    display(cluster_profile_numeric)

for col in categorical_profile_cols[:3]:
    display(pd.crosstab(clustered_df['cluster'], clustered_df[col], normalize='index').round(2))


## Business Interpretation

Use this structure for each cluster:

- **Cluster name:** Example: high-value loyalists, price-sensitive browsers, enterprise evaluators
- **Defining traits:** Which features make this group different?
- **Business value:** Why does this cluster matter?
- **Recommended action:** What should marketing, product, or sales do differently?
- **Risk:** What could go wrong if the segment definition is unstable?
